# Built with DINOv3

This notebook uses weights derived from Meta's **DINOv3**. DINOv3 is distributed under the DINOv3 License, included in the attached dataset as `DINOV3_LICENSE.md`.

Other pretrained weights used here carry their own separate licences.

In [ ]:
import sys, glob, json, subprocess, traceback, importlib.util
from pathlib import Path
hits = sorted(glob.glob('/kaggle/input/**/shared_code.py', recursive=True))
if not hits:
    raise SystemExit('shared_code.py not found. tree: '
                     + str(sorted(glob.glob('/kaggle/input/**/*', recursive=True))[:200]))
W = Path(hits[0]).parent
sys.path.insert(0, str(W))
# Kaggle sometimes mounts the PREVIOUS dataset version right after an upload (cost a 43 min
# stress run on 1 Oct). Stop in seconds instead; the publisher sees this and pushes again.
import hashlib
_eng = hashlib.sha1((W / 'shared_code.py').read_bytes()).hexdigest()
if _eng != '3ac3538a066755393e757b084776e4dd5cbcd628':
    Path('engine_mismatch.txt').write_text(_eng)
    raise RuntimeError('ENGINE MISMATCH: mounted %s, expected 3ac3538a066755393e757b084776e4dd5cbcd628' % _eng)
print('engine', _eng)
subs = sorted(glob.glob('/kaggle/input/**/sample_submission.csv', recursive=True))
if not subs:
    raise SystemExit('competition data not found under /kaggle/input')
COMP = Path(subs[0]).parent
MEMBERS = json.loads((W / 'members.json').read_text())
BLEND = None
_bw = W / 'blend_weights.json'
if _bw.exists():
    BLEND = json.loads(_bw.read_text())
    print('per-class blend weights found for', len(BLEND['members']), 'members')
else:
    print('no blend_weights.json -> flat average')
print('weights:', W, '| comp:', COMP)
print('members:', [m['name'] for m in MEMBERS])

In [ ]:
# the test set mixes JPEG Lossless and JPEG 2000 in with the raw series and pydicom
# reads neither on its own. no internet here, so the wheels ride along in the dataset.
wheels = sorted(glob.glob(str(W / '*.whl')))
try:
    import pylibjpeg, openjpeg, libjpeg
    print('dicom decoders already present')
except ImportError:
    if wheels:
        r = subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index',
                            '--no-deps', '-q', *wheels], capture_output=True, text=True)
        print('installed decoders rc=', r.returncode, r.stderr[-300:])
    else:
        print('WARNING: no decoder wheels, compressed series will fail')

In [ ]:
import shared_code as sc, pandas as pd, torch
series = pd.read_csv(COMP / 'test_series.csv')
sub = pd.read_csv(COMP / 'sample_submission.csv')
TASKS = [c for c in sub.columns if c != 'StudyInstanceUID']
print(len(sub), 'studies to score')

def bail(why):
    # always leave a valid file behind: 0.5 everywhere scores about 0.5, while no file
    # at all scores nothing and still costs one of the five daily attempts
    traceback.print_exc()
    print('FALLING BACK TO 0.5:', why)
    out = sub.copy()
    out[TASKS] = 0.5
    out.to_csv('submission.csv', index=False)

bail('nothing has run yet')          # overwritten the moment real work lands

In [ ]:
# member 0 owns the loader geometry; the others only re-normalise the same batch
try:
    CFGS = [json.loads((W / ('m%d_config.json' % m['i'])).read_text()) for m in MEMBERS]
    idx, files_by_series = sc.scan_dicom_index(COMP / 'test_series', series)
    plan = sc.plan_slots(idx, CFGS[0])
    print(len(plan), 'studies planned of', len(sub))
except Exception:
    bail('scan/plan failed'); raise SystemExit(0)

In [ ]:
# each member's model_def under its OWN module name - every run's train.py defines a
# class called SliceModel, so one shared `model_def` would hand every member the same
# architecture and silently score the wrong thing
def _load_def(i):
    spec = importlib.util.spec_from_file_location('m%d_def' % i, W / ('m%d_def.py' % i))
    mod = importlib.util.module_from_spec(spec)
    sys.modules['m%d_def' % i] = mod
    spec.loader.exec_module(mod)
    return mod

# Load weights whichever side of torch.compile they were saved from. The notebook does
# not import shared_code, so this mirrors its helper: strip a leading '_orig_mod.' (left
# by a checkpoint written from a compiled model) and load into the unwrapped module.
def _load_model_state(model, sd):
    if any(k.startswith('_orig_mod.') for k in sd):
        sd = {(k[len('_orig_mod.'):] if k.startswith('_orig_mod.') else k): v
              for k, v in sd.items()}
    return getattr(model, '_orig_mod', model).load_state_dict(sd)

# SPECS say how to rebuild a member from nothing. The second T4 is driven by a separate
# PROCESS, because threads share one interpreter lock and measured 2x slower per card, and
# a built model holding CUDA tensors cannot be pickled to a child. The child rebuilds its
# own members from these same three files. Plain data only, so it survives the spawn.
SPECS = []
try:
    MODELS = []
    for m, cfg in zip(MEMBERS, CFGS):
        i = m['i']
        SPECS.append({'i': int(i), 'dir': str(W), 'cfg': cfg})
        model = _load_def(i).build_model(cfg)
        ck = torch.load(W / ('m%d_model.pt' % i), map_location='cpu', weights_only=False)
        _load_model_state(model, ck['model'])
        MODELS.append(model.eval())
        print('m%d %-30s epoch %s score %s' % (i, m['name'], ck.get('epoch'), ck.get('score')))
except Exception:
    bail('model load failed'); raise SystemExit(0)

In [ ]:
# STRESS repeats the public studies to imitate the hidden set's size, so the
# per-study rate is MARGINAL rather than dominated by one-time warm-up. It also
# suppresses submission.csv, because the repeated rows are not a real submission.
# 0 = a normal run.
STRESS = 128

todo = sub[['StudyInstanceUID']]
if STRESS:
    reps = -(-STRESS // len(todo))
    todo = pd.concat([todo] * reps, ignore_index=True).head(STRESS)
    print('STRESS MODE:', len(todo), 'studies')

# ONE decode pass shared by every member. Decoding is ~9s/study against <1s of GPU,
# so looping predict_dicom per member would be ~4x9s = over the 9h cap; this is
# ~9s + N x <1s. Averaging happens in logit space, as measured offline.
import time as _time
_t0 = _time.time()
try:
    # reorder the saved matrix into MEMBERS order, and into the task order this
    # notebook actually submits in - both are checked, because a silent mismatch here
    # blends the wrong model into the wrong finding and still writes a valid-looking file
    _W = None
    if BLEND:
        _names = [m['name'] for m in MEMBERS]
        assert _names == BLEND['members'], (
            'member order differs: %s vs %s' % (_names, BLEND['members']))
        _ti = [BLEND['tasks'].index(t) for t in TASKS]
        _W = [[row[j] for j in _ti] for row in BLEND['matrix']]
        print('using per-class blend weights')
    # _grouped, not the plain one: members that disagree on loader geometry (size,
    # mm_per_px, n_slices, slots, windowing) cannot share a decode pass, so it decodes once
    # per geometry GROUP and blends at the end. With every member on one geometry this does
    # exactly what predict_dicom_ensemble did, in one pass, so it is safe as the default.
    pred = sc.predict_dicom_ensemble_grouped(CFGS, MODELS, todo, plan, idx, files_by_series,
                                             batch_size=1, weights=_W, model_specs=SPECS)
    # the 9h cap is enforced on the HIDDEN test set, which is larger than what runs
    # here, so the number that has to be extrapolated is s/study, not the total
    _el = _time.time() - _t0
    _rate = _el / max(len(todo), 1)
    print('INFERENCE  %.1f s for %d studies  =  %.2f s/study  |  9h cap fits ~%d studies'
          % (_el, len(todo), _rate, int(9 * 3600 / max(_rate, 1e-9))))
    # timing.json is written by predict_dicom_ensemble and holds the decode/per-member
    # split. Echo it so it is in the log too, then leave the file as a kernel output for
    # runtime_budget.py to ingest.
    try:
        print('TIMING ' + open('timing.json').read().replace(chr(10), ' '))
    except Exception:
        pass
    if not STRESS:
        out = sub[['StudyInstanceUID']].merge(pred, on='StudyInstanceUID', how='left')
        out[TASKS] = out[TASKS].astype(float).fillna(0.5).clip(0, 1)
        assert len(out) == len(sub), '%d rows for %d studies' % (len(out), len(sub))
        out.to_csv('submission.csv', index=False)
        print(out[TASKS].describe().T[['mean', 'min', 'max']])
except Exception:
    bail('inference failed')
pd.read_csv('submission.csv').head()